In [18]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10

In [19]:
# ============================================
# LOAD CIFAR-10 DATASET (TRAINING)
# ============================================

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("CIFAR Train Shape:", x_train.shape)
print("CIFAR Test Shape :", x_test.shape)

CIFAR Train Shape: (50000, 32, 32, 3)
CIFAR Test Shape : (10000, 32, 32, 3)


In [20]:
# ============================================
# NORMALIZE CIFAR DATA
# ============================================

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = y_train.flatten()
y_test = y_test.flatten()

In [21]:
# ============================================
# LOAD CINIC-10 TEST DATASET
# ============================================

cinic_test = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/mengcius/cinic10/test',
    image_size=(32,32),
    batch_size=64,
    shuffle=False
)

Found 90000 files belonging to 10 classes.


In [22]:
# ============================================
# NORMALIZE CINIC DATA
# ============================================

cinic_test = cinic_test.map(
    lambda x, y: (tf.cast(x, tf.float32) / 255.0, y)
)

In [23]:
# ============================================
# DATA AUGMENTATION
# ============================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [24]:
# ============================================
# DENSE BLOCK
# ============================================

def dense_block(x, num_layers, growth_rate):

    for _ in range(num_layers):

        y = layers.BatchNormalization()(x)
        y = layers.ReLU()(y)

        y = layers.Conv2D(
            4 * growth_rate,
            (1,1),
            padding='same',
            use_bias=False
        )(y)

        y = layers.BatchNormalization()(y)
        y = layers.ReLU()(y)

        y = layers.Conv2D(
            growth_rate,
            (3,3),
            padding='same',
            use_bias=False
        )(y)

        x = layers.Concatenate()([x, y])

    return x

In [25]:
# ============================================
# TRANSITION LAYER
# ============================================

def transition_layer(x, reduction=0.5):

    filters = int(x.shape[-1] * reduction)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(
        filters,
        (1,1),
        padding='same',
        use_bias=False
    )(x)

    x = layers.AveragePooling2D((2,2), strides=2)(x)

    return x

In [26]:
# ============================================
# BUILD DENSENET MODEL
# ============================================

def build_densenet(input_shape=(32,32,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    x = data_augmentation(inputs)

    # Initial Conv
    x = layers.Conv2D(
        64,
        (3,3),
        padding='same',
        use_bias=False
    )(x)

    # Dense Block 1
    x = dense_block(x, num_layers=4, growth_rate=12)
    x = transition_layer(x)

    # Dense Block 2
    x = dense_block(x, num_layers=4, growth_rate=12)
    x = transition_layer(x)

    # Dense Block 3
    x = dense_block(x, num_layers=4, growth_rate=12)

    # Final Layers
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        num_classes,
        activation='softmax'
    )(x)

    model = models.Model(inputs, outputs)

    return model

In [27]:
# ============================================
# CREATE MODEL
# ============================================

model = build_densenet()

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 32, 32, 3) │          0 │ input_layer_2[0]… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_27 (Conv2D)  │ (None, 32, 32,    │      1,728 │ sequential_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_27[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_27 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_28 (Conv2D)  │ (None, 32, 32,    │      3,072 │ re_lu_27[0][0]    │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        192 │ conv2d_28[0][0]   │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_28 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_29 (Conv2D)  │ (None, 32, 32,    │      5,184 │ re_lu_28[0][0]    │
│                     │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_12      │ (None, 32, 32,    │          0 │ conv2d_27[0][0],  │
│ (Concatenate)       │ 76)               │            │ conv2d_29[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        304 │ concatenate_12[0… │
│ (BatchNormalizatio… │ 76)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_29 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 76)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_30 (Conv2D)  │ (None, 32, 32,    │      3,648 │ re_lu_29[0][0]    │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        192 │ conv2d_30[0][0]   │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_30 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_31 (Conv2D)  │ (None, 32, 32,    │      5,184 │ re_lu_30[0][0]    │
│                     │ 12)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_13      │ (None, 32, 32,    │          0 │ concatenate_12[0

 Total params: 140,410 (548.48 KB)

 Trainable params: 136,818 (534.45 KB)

 Non-trainable params: 3,592 (14.03 KB)

In [28]:
# ============================================
# COMPILE MODEL
# ============================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [29]:
# ============================================
# TRAIN ON CIFAR-10
# ============================================

history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=40,
    validation_data=(x_test, y_test)
)

Epoch 1/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 75s 79ms/step - accuracy: 0.3225 - loss: 1.8115 - val_accuracy: 0.4346 - val_loss: 1.5620
Epoch 2/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 60s 77ms/step - accuracy: 0.5263 - loss: 1.3139 - val_accuracy: 0.5047 - val_loss: 1.4762
Epoch 3/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 59s 76ms/step - accuracy: 0.5923 - loss: 1.1402 - val_accuracy: 0.4447 - val_loss: 1.6026
Epoch 4/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 59s 76ms/step - accuracy: 0.6294 - loss: 1.0408 - val_accuracy: 0.5874 - val_loss: 1.1720
Epoch 5/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 59s 76ms/step - accuracy: 0.6579 - loss: 0.9817 - val_accuracy: 0.5924 - val_loss: 1.2016
Epoch 6/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 59s 76ms/step - accuracy: 0.6724 - loss: 0.9270 - val_accuracy: 0.6655 - val_loss: 0.9643
Epoch 7/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 59s 76ms/step - accuracy: 0.6907 - loss: 0.8863 - val_accuracy: 0.5800 - val_loss: 1.3145
Epoch 8/40
782/782 ━━━━━━━━━━━━━━━━━━━━ 60s 77ms/step - accuracy: 0.7120 - loss: 0.8288 - 

In [32]:
# ============================================
# TEST ON CINIC-10
# ============================================

test_loss, test_acc = model.evaluate(cinic_test)

print("CIFAR → CINIC DenseNet Accuracy:", test_acc)

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 33s 23ms/step - accuracy: 0.6318 - loss: 1.3467
CIFAR → CINIC DenseNet Accuracy: 0.5952000021934509


In [33]:
# ============================================
# SAVE MODEL
# ============================================

model.save('/kaggle/working/cifar_to_cinic_densenet.keras')